<a href="https://colab.research.google.com/github/Charan290904/Charan-ML-2/blob/main/SimpleFoil.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Simple FOIL implementation for a small dataset

# ---------------------------------------------------------
# Dataset
# ---------------------------------------------------------

# Facts are represented as:
# predicate(subject, object)

facts = {
    "parent": {
        ("alice", "bob"),
        ("alice", "carol"),
        ("bob", "david"),
        ("carol", "emma")
    },

    "female": {
        ("alice",),
        ("carol",),
        ("emma",)
    },

    "male": {
        ("bob",),
        ("david",)
    }
}

# Target predicate:
# grandparent(X, Y)

# Positive examples
positive_examples = {
    ("alice", "david"),
    ("alice", "emma")
}

# Negative examples
negative_examples = {
    ("alice", "bob"),
    ("bob", "emma"),
    ("carol", "david")
}


# ---------------------------------------------------------
# Helper functions
# ---------------------------------------------------------

def parent(x, y):
    return (x, y) in facts["parent"]


def female(x):
    return (x,) in facts["female"]


def male(x):
    return (x,) in facts["male"]


# ---------------------------------------------------------
# Candidate rules
# ---------------------------------------------------------

# For this small example, we manually define possible
# literals that FOIL can consider.

candidate_literals = [
    "parent(X, Z)",
    "parent(Z, Y)",
    "female(X)",
    "female(Y)",
    "male(X)",
    "male(Y)"
]


def satisfies_literal(example, literal):
    """
    Check whether a literal is true for an example.

    For the grandparent example:
        X = first element
        Y = second element

    Z is searched for automatically.
    """

    X, Y = example

    if literal == "female(X)":
        return female(X)

    if literal == "female(Y)":
        return female(Y)

    if literal == "male(X)":
        return male(X)

    if literal == "male(Y)":
        return male(Y)

    # parent(X, Z) AND parent(Z, Y)
    if literal == "parent(X, Z)":
        return any(parent(X, z) for z in get_people())

    if literal == "parent(Z, Y)":
        return any(parent(z, Y) for z in get_people())

    return False


def get_people():
    people = set()

    for x, y in facts["parent"]:
        people.add(x)
        people.add(y)

    return people


# ---------------------------------------------------------
# Rule representation
# ---------------------------------------------------------

class Rule:
    def __init__(self):
        self.conditions = []

    def add_condition(self, condition):
        self.conditions.append(condition)

    def covers(self, example):
        """
        Check whether the rule covers an example.
        """

        X, Y = example

        # Special handling for the two conditions needed
        # for grandparent(X,Y).

        has_parent_xz = "parent(X, Z)" in self.conditions
        has_parent_zy = "parent(Z, Y)" in self.conditions

        if has_parent_xz and has_parent_zy:

            # There must exist the SAME Z satisfying both.
            for z in get_people():
                if parent(X, z) and parent(z, Y):
                    return True

            return False

        # If only parent(X,Z) exists
        if has_parent_xz:
            if not any(parent(X, z) for z in get_people()):
                return False

        # If only parent(Z,Y) exists
        if has_parent_zy:
            if not any(parent(z, Y) for z in get_people()):
                return False

        # Other conditions
        if "female(X)" in self.conditions:
            if not female(X):
                return False

        if "female(Y)" in self.conditions:
            if not female(Y):
                return False

        if "male(X)" in self.conditions:
            if not male(X):
                return False

        if "male(Y)" in self.conditions:
            if not male(Y):
                return False

        return True

    def __str__(self):
        if not self.conditions:
            return "grandparent(X, Y) :- TRUE"

        return "grandparent(X, Y) :- " + ", ".join(self.conditions)


# ---------------------------------------------------------
# FOIL Gain
# ---------------------------------------------------------

def foil_gain(rule, literal, pos, neg):
    """
    Simplified FOIL Gain.

    We calculate how many positive and negative examples
    remain after adding a literal.

    gain = positive_remaining / total_remaining
    """

    old_covered_pos = [
        e for e in pos
        if rule.covers(e)
    ]

    old_covered_neg = [
        e for e in neg
        if rule.covers(e)
    ]

    # Temporarily add literal
    rule.add_condition(literal)

    new_covered_pos = [
        e for e in pos
        if rule.covers(e)
    ]

    new_covered_neg = [
        e for e in neg
        if rule.covers(e)
    ]

    # Remove temporary literal
    rule.conditions.pop()

    old_total = len(old_covered_pos) + len(old_covered_neg)
    new_total = len(new_covered_pos) + len(new_covered_neg)

    if new_total == 0:
        return 0

    old_probability = (
        len(old_covered_pos) / old_total
        if old_total else 0
    )

    new_probability = (
        len(new_covered_pos) / new_total
        if new_total else 0
    )

    return new_probability - old_probability


# ---------------------------------------------------------
# FOIL algorithm
# ---------------------------------------------------------

def foil(pos, neg):

    learned_rules = []

    pos = set(pos)
    neg = set(neg)

    while pos:

        rule = Rule()
        rule_neg = set(neg)

        print("\nStarting new rule")

        while rule_neg:

            best_literal = None
            best_gain = -1

            for literal in candidate_literals:

                if literal in rule.conditions:
                    continue

                gain = foil_gain(
                    rule,
                    literal,
                    pos,
                    rule_neg
                )

                print(
                    f"  Candidate: {literal:15} "
                    f"Gain = {gain:.3f}"
                )

                if gain > best_gain:
                    best_gain = gain
                    best_literal = literal

            if best_literal is None:
                break

            rule.add_condition(best_literal)

            # Keep only negative examples still covered
            rule_neg = {
                example
                for example in rule_neg
                if rule.covers(example)
            }

            print(
                f"  Added: {best_literal}"
            )

        learned_rules.append(rule)

        # Remove positive examples covered by this rule
        covered_pos = {
            example
            for example in pos
            if rule.covers(example)
        }

        pos -= covered_pos

        print("Learned:", rule)
        print("Covered positive examples:", covered_pos)

    return learned_rules


# ---------------------------------------------------------
# Run FOIL
# ---------------------------------------------------------

rules = foil(
    positive_examples,
    negative_examples
)

print("\n==============================")
print("FINAL LEARNED RULES")
print("==============================")

for rule in rules:
    print(rule)






Starting new rule
  Candidate: parent(X, Z)    Gain = 0.000
  Candidate: parent(Z, Y)    Gain = 0.000
  Candidate: female(X)       Gain = 0.100
  Candidate: female(Y)       Gain = 0.100
  Candidate: male(X)         Gain = -0.400
  Candidate: male(Y)         Gain = -0.067
  Added: female(X)
  Candidate: parent(X, Z)    Gain = 0.000
  Candidate: parent(Z, Y)    Gain = 0.000
  Candidate: female(Y)       Gain = 0.500
  Candidate: male(X)         Gain = 0.000
  Candidate: male(Y)         Gain = -0.167
  Added: female(Y)
Learned: grandparent(X, Y) :- female(X), female(Y)
Covered positive examples: {('alice', 'emma')}

Starting new rule
  Candidate: parent(X, Z)    Gain = 0.000
  Candidate: parent(Z, Y)    Gain = 0.000
  Candidate: female(X)       Gain = 0.083
  Candidate: female(Y)       Gain = -0.250
  Candidate: male(X)         Gain = -0.250
  Candidate: male(Y)         Gain = 0.083
  Added: female(X)
  Candidate: parent(X, Z)    Gain = 0.000
  Candidate: parent(Z, Y)    Gain = 0.000
  Ca